# Strategy Comparison

**Strategy A:** Buy $100 every red monthly candle + progressive reinvestment (1% start, +2%/buy, cap 50%), sell 70% at each new ATH
**Strategy B:** Buy $100 every red monthly candle, **never sell** (pure BTC accumulation)
**Strategy C:** Buy $100 **every** month regardless of candle color, **never sell** (classic DCA)

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
# Fetch data once

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
# === STRATEGY A: Progressive Reinvestment with 70% sells ===

START_PCT = 1.0; INCREMENT = 2.0; CAP = 50.0; BASE = 100.0

btc_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0
reinvest_pct = START_PCT
records_a = []

for i, row in df.iterrows():
    close = row['close']
    if close < row['open']:
        extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
        total_usdt = BASE + extra
        btc_held += total_usdt / close
        total_invested += BASE
        usdt_reserve -= extra

    prev_highest = highest_close
    if close > highest_close:
        highest_close = close

    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        usdt_reserve += sell_btc * close
        btc_held -= sell_btc
        reinvest_pct = START_PCT
    else:
        if reinvest_pct < CAP:
            reinvest_pct = min(CAP, reinvest_pct + INCREMENT)

    records_a.append(btc_held * close + usdt_reserve)

print(f"Strategy A done: ${records_a[-1]:.0f}")

In [ ]:
# === STRATEGY B: Buy $100 every red candle, never sell ===

btc_held = 0.0; total_invested = 0.0
records_b = []

for i, row in df.iterrows():
    close = row['close']
    if close < row['open']:
        btc_held += BASE / close
        total_invested += BASE
    records_b.append(btc_held * close)

print(f"Strategy B done: ${records_b[-1]:.0f}")

In [ ]:
# === STRATEGY C: Buy $100 every month (classic DCA), never sell ===

btc_held = 0.0; total_invested = 0.0
records_c = []

for i, row in df.iterrows():
    close = row['close']
    btc_held += BASE / close
    total_invested += BASE
    records_c.append(btc_held * close)

final_c_invested = BASE * len(df)
print(f"Strategy C done: ${records_c[-1]:.0f}")

In [ ]:
# === COMPARISON SUMMARY ===

a_final = records_a[-1]
b_final = records_b[-1]
c_final = records_c[-1]

# Compute metrics for all three
strategies = []
for name, rec, inv in [
    ('A: Prog+Sell', records_a, 5100),
    ('B: Red-only HODL', records_b, sum(BASE for r in df.iterrows() if r[1]['close'] < r[1]['open']) if False else None),
    ('C: Monthly DCA', records_c, BASE * len(df))
]:
    pass

# Recompute properly
inv_a = 0; btc_a = 0; usdt_a = 0; hc_a = 0; rp = START_PCT
for i, row in df.iterrows():
    c = row['close']
    if c < row['open']:
        ex = usdt_a * (rp / 100.0) if usdt_a > 0 else 0
        btc_a += (BASE + ex) / c
        inv_a += BASE
        usdt_a -= ex
    ph = hc_a
    if c > hc_a: hc_a = c
    if btc_a > 0.000001 and c > ph:
        sb = btc_a * 0.7
        usdt_a += sb * c
        btc_a -= sb
        rp = START_PCT
    else:
        if rp < CAP: rp = min(CAP, rp + INCREMENT)

inv_b = 0; btc_b = 0
for i, row in df.iterrows():
    if row['close'] < row['open']:
        btc_b += BASE / row['close']
        inv_b += BASE

inv_c = BASE * len(df); btc_c = 0
for i, row in df.iterrows():
    btc_c += BASE / row['close']

last_close = df.iloc[-1]['close']

a_val = btc_a * last_close + usdt_a
b_val = btc_b * last_close
c_val = btc_c * last_close

def metrics(records, invested):
    s = pd.Series(records)
    ret = (s.iloc[-1] / invested - 1) * 100
    rm = s.cummax()
    dd = ((rm - s) / rm).max() * 100
    mr = s.pct_change().dropna()
    mr = mr[mr.index >= 12]
    sh = (mr.mean() / mr.std()) * np.sqrt(12) if mr.std() > 0 else 0
    ps = mr[mr > 0].sum(); ns = mr[mr < 0].abs().sum()
    pf = ps / ns if ns > 0 else float('inf')
    ann = (1 + ret / 100) ** (12 / len(df)) - 1
    cal = ann / (dd / 100) if dd > 0 else 0
    return ret, dd, sh, cal, pf

def dd_series(records):
    s = pd.Series(records)
    rm = s.cummax()
    return ((rm - s) / rm) * 100

r_a, d_a, sh_a, ca_a, pf_a = metrics(records_a, inv_a)
r_b, d_b, sh_b, ca_b, pf_b = metrics(records_b, inv_b)
r_c, d_c, sh_c, ca_c, pf_c = metrics(records_c, inv_c)

print("="*72)
print(f"{'':>25s} {'A: Prog+Sell':>14s} {'B: Red HODL':>14s} {'C: Monthly DCA':>14s}")
print("="*72)
print(f"{'Total invested':>25s} {f'${inv_a:,}':>14s} {f'${inv_b:,}':>14s} {f'${inv_c:,}':>14s}")
print(f"{'Portfolio value':>25s} {f'${a_val:,.0f}':>14s} {f'${b_val:,.0f}':>14s} {f'${c_val:,.0f}':>14s}")
print(f"{'Return':>25s} {f'{r_a:.1f}%':>14s} {f'{r_b:.1f}%':>14s} {f'{r_c:.1f}%':>14s}")
print(f"{'Max DD':>25s} {f'{d_a:.1f}%':>14s} {f'{d_b:.1f}%':>14s} {f'{d_c:.1f}%':>14s}")
print(f"{'Sharpe':>25s} {f'{sh_a:.2f}':>14s} {f'{sh_b:.2f}':>14s} {f'{sh_c:.2f}':>14s}")
print(f"{'Calmar':>25s} {f'{ca_a:.2f}':>14s} {f'{ca_b:.2f}':>14s} {f'{ca_c:.2f}':>14s}")
print(f"{'Profit Factor':>25s} {f'{pf_a:.2f}':>14s} {f'{pf_b:.2f}':>14s} {f'{pf_c:.2f}':>14s}")
print(f"{'BTC held':>25s} {f'{btc_a:.6f}':>14s} {f'{btc_b:.6f}':>14s} {f'{btc_c:.6f}':>14s}")
print(f"{'USDT reserve':>25s} {f'${usdt_a:,.0f}':>14s} {'$0':>14s} {'$0':>14s}")

In [ ]:
# === VALUE OVER TIME CHART ===

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

dates = df['date']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]})

# Top: portfolio values
ax1.plot(dates, records_a, color='#2563eb', linewidth=2, label=f'A: Prog+Sell (${a_val:,.0f})')
ax1.plot(dates, records_b, color='#16a34a', linewidth=2, label=f'B: Red-only HODL (${b_val:,.0f})')
ax1.plot(dates, records_c, color='#dc2626', linewidth=1.5, linestyle='--', label=f'C: Monthly DCA (${c_val:,.0f})')

# Mark sell events for Strategy A
sells = []
hc = 0
for i, row in df.iterrows():
    if row['close'] > hc:
        hc = row['close']
        if i > 0:  # skip first month
            # Check if there was BTC to sell
            pass

# Recompute sell dates
btc = 0; usdt = 0; hc = 0; rp = START_PCT
sell_dates = []
for i, row in df.iterrows():
    c = row['close']; ph = hc
    if c > hc: hc = c
    if btc > 0.000001 and c > ph:
        sell_dates.append(row['date'])
        sb = btc * 0.7
        usdt += sb * c
        btc -= sb
        rp = START_PCT
    if c < row['open']:
        ex = usdt * (rp / 100.0) if usdt > 0 else 0
        btc += (BASE + ex) / c
        usdt -= ex
        rp = min(CAP, rp + INCREMENT)

for sd in sell_dates:
    ax1.axvline(x=sd, color='#f59e0b', alpha=0.3, linewidth=1, linestyle=':')

ax1.set_title('Strategy Comparison — $100/mo Base Investment', fontsize=14, fontweight='bold')
ax1.set_ylabel('Portfolio Value (USDT)')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Bottom: drawdown for each
dd_a = dd_series(records_a)
dd_b = dd_series(records_b)
dd_c = dd_series(records_c)

ax2.fill_between(dates, 0, dd_a, color='#2563eb', alpha=0.2, label=f'A: Max DD {d_a:.1f}%')
ax2.plot(dates, dd_a, color='#2563eb', linewidth=1)
ax2.fill_between(dates, 0, dd_b, color='#16a34a', alpha=0.15, label=f'B: Max DD {d_b:.1f}%')
ax2.plot(dates, dd_b, color='#16a34a', linewidth=1)
ax2.fill_between(dates, 0, dd_c, color='#dc2626', alpha=0.1, label=f'C: Max DD {d_c:.1f}%')
ax2.plot(dates, dd_c, color='#dc2626', linewidth=1, linestyle='--')

ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.legend(loc='lower left', fontsize=9, ncol=3)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-comparison.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-comparison.png')
plt.show()

In [ ]:
# === BTC ACCUMULATION OVER TIME ===

btc_a_hist = []; btc_b_hist = []; btc_c_hist = []

btc = 0; usdt = 0; hc = 0; rp = START_PCT
for i, row in df.iterrows():
    c = row['close']; ph = hc
    if c > hc: hc = c
    if btc > 0.000001 and c > ph:
        btc -= btc * 0.7
        rp = START_PCT
    if c < row['open']:
        ex = usdt * (rp / 100.0) if usdt > 0 else 0
        btc += (BASE + ex) / c
        usdt -= ex
        rp = min(CAP, rp + INCREMENT)
    btc_a_hist.append(btc)

btc = 0
for i, row in df.iterrows():
    if row['close'] < row['open']:
        btc += BASE / row['close']
    btc_b_hist.append(btc)

btc = 0
for i, row in df.iterrows():
    btc += BASE / row['close']
    btc_c_hist.append(btc)

fig, ax = plt.subplots(figsize=(14, 5))
ax.step(dates, btc_a_hist, where='post', color='#2563eb', linewidth=2, label=f'A: Prog+Sell ({btc_a_hist[-1]:.4f} BTC)')
ax.step(dates, btc_b_hist, where='post', color='#16a34a', linewidth=2, label=f'B: Red-only HODL ({btc_b_hist[-1]:.4f} BTC)')
ax.step(dates, btc_c_hist, where='post', color='#dc2626', linewidth=1.5, linestyle='--', label=f'C: Monthly DCA ({btc_c_hist[-1]:.4f} BTC)')

for sd in sell_dates:
    ax.axvline(x=sd, color='#f59e0b', alpha=0.3, linewidth=1, linestyle=':')

ax.set_title('BTC Holdings Over Time', fontsize=13, fontweight='bold')
ax.set_ylabel('BTC')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-comparison-btc.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-comparison-btc.png')
plt.show()